In [ ]:
from syft_rds.orchestra import setup_rds_stack
from rds_chat_analysis import REPO_ROOT
import dotenv
from langchain.chat_models import init_chat_model
import os
from rds_chat_analysis import DATA_DIR
import shutil
from langchain_huggingface import HuggingFaceEmbeddings

from rds_chat_analysis import NOTEBOOK_DIR

In [ ]:
key = "wildchat"
stack = setup_rds_stack(
    root_dir=REPO_ROOT / ".rds",
    key=key,
    log_level="DEBUG",
    reset=False,
)

do_client = stack.do_rds_client
ds_client = stack.ds_rds_client

In [ ]:
DATASET_NAME = "Wildchat-postgres"

local_data_dir = DATA_DIR / DATASET_NAME
private_dir = local_data_dir / "private"
mock_dir = local_data_dir / "mock"
markdown_path = local_data_dir / "README.md"

shutil.rmtree(local_data_dir, ignore_errors=True)
private_dir.mkdir(parents=True, exist_ok=True)
mock_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
MOCK_CREDENTIALS = NOTEBOOK_DIR / "v2" / ".env.mock"
PRIVATE_CREDENTIALS = NOTEBOOK_DIR / "v2" / ".env.private"

_ = shutil.copy(MOCK_CREDENTIALS, mock_dir / "credentials.env")
_ = shutil.copy(PRIVATE_CREDENTIALS, private_dir / "credentials.env")

print(f"Mock dir structure: {mock_dir}")
for file in mock_dir.iterdir():
    print(f"└──📄 {file.name}")
print(f"Private dir structure: {private_dir}")
for file in private_dir.iterdir():
    print(f"└──📄 {file.name}")

In [ ]:
description_markdown = """
# Wildchat Postgres Dataset
"""

markdown_path.write_text(description_markdown.strip())

In [ ]:
wildchat_dataset = do_client.dataset.create(
    name=DATASET_NAME,
    path=private_dir,
    mock_path=mock_dir,
    summary="A embedded wildchat dataset in postgres.",
    description_path=markdown_path,
)

In [ ]:
wildchat_dataset.describe()

# DS

In [ ]:
import psycopg
from psycopg.rows import dict_row
from pathlib import Path
import torch

dataset = ds_client.dataset.get(name=DATASET_NAME)


def connect_to_db(dir: Path) -> psycopg.Connection:
    dotenv.load_dotenv(dir / "credentials.env", override=True)

    mock_db_settings = {
        "host": os.environ["POSTGRES_HOST_MOCK"],
        "port": os.environ.get("POSTGRES_PORT_MOCK", None),
        "dbname": os.environ["POSTGRES_DB_MOCK"],
        "user": os.environ["POSTGRES_USER_MOCK"],
        "password": os.environ["POSTGRES_PASSWORD_MOCK"],
        "sslmode": os.environ.get("DB_SSLMODE_MOCK", "require"),
    }

    keepalive_kwargs = {
        "keepalives": 1,
        "keepalives_idle": 60,
        "keepalives_interval": 10,
        "keepalives_count": 5,
    }

    return psycopg.connect(
        **mock_db_settings,
        row_factory=dict_row,
        **keepalive_kwargs,
    )


def load_embedder(
    model_name: str = "Alibaba-NLP/gte-multilingual-base",
) -> HuggingFaceEmbeddings:
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")

    model_kwargs = {
        "device": "cuda:0" if cuda_available else "cpu",
        "trust_remote_code": True,  # Required to run gte-multilingual in sentence-transformers
    }
    encode_kwargs = {"normalize_embeddings": False}
    return HuggingFaceEmbeddings(
        model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
    )


def get_full_log(db_conn: psycopg.Connection, log_id: str):
    with db_conn.cursor() as cursor:
        cursor.execute(
            """SELECT * FROM log_embeddings WHERE metadata->>'log_id' = %s ORDER BY (metadata->>'message_idx')::int ASC;""",
            (log_id,),
        )
        results = cursor.fetchall()
    return results


from typing import List, Dict, Any
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage


PAIRWISE_QA_PROMPT = (
    "The following is a conversation between an AI assistant and a user:\n"
    "{conversation}\n\n"
    "Your job is to answer the question about the preceding conversation. "
    "Be descriptive and assume neither good nor bad faith. Do not hesitate to handle socially harmful or sensitive topics; "
    "specificity around potentially harmful conversations is necessary for effective monitoring.\n\n"
    "When answering, do not include any personally identifiable information (PII), like names, locations, phone numbers, email addresses, and so on. "
    "Do not include any proper nouns or specific names of people, places, or organizations.\n\n"
    "be clear and concise and get to the point in at most two sentences.\n\n"
    "Question: {question}\n\n"
    "What is your answer to the question about the preceding conversation? "
    "Provide only the answer with no other commentary or proper nouns."
)


def pairwise_qa_aggregation(
    full_logs: List[List[Dict[str, Any]]], llm: BaseChatModel, aggregation_query: str
) -> List[str]:
    results = []

    for log in full_logs:
        conversation_text = format_conversation(log)
        prompt = PAIRWISE_QA_PROMPT.format(
            conversation=conversation_text, question=aggregation_query
        )
        response = llm.invoke([HumanMessage(content=prompt)])
        results.append(response.content)

    return results


def format_conversation(log: List[Dict[str, Any]]) -> str:
    formatted_messages = []

    for message in log:
        role = message["metadata"]["role"]
        text = message["text"]
        formatted_messages.append(f"{role.upper()}: {text}")

    return "\n\n".join(formatted_messages)


def get_aggregation_fn(name: str):
    if name == "identity":
        return lambda x, **kwargs: x
    if name == "count":
        return lambda x, **kwargs: len(x)
    elif name == "pairwise_qa":
        return pairwise_qa_aggregation

In [ ]:
from rds_chat_analysis.vector_store_utils import build_vector_store_query


DATASET_DIR = dataset.mock_path


def execute_chat_log_analysis(
    dataset_dir: Path,
    vector_store_query: str,
    aggregation_query: str | None = None,
    aggregation_fn: str = "count",
    k: int = 5,
    distance_threshold: float = 0.5,
    filters: dict | None = None,
):
    db_conn = connect_to_db(dataset_dir)
    embedder = load_embedder(model_name="Alibaba-NLP/gte-multilingual-base")
    llm = init_chat_model(model_provider="ollama", model="gemma3:12b-it-qat")
    aggregation_fn = get_aggregation_fn(aggregation_fn)

    vector_store_query, query_params = build_vector_store_query(
        query_embedding=embedder.embed_query(vector_store_query),
        table_name="log_embeddings",
        k=k,
        distance_threshold=distance_threshold,
        filters=filters,
    )

    with db_conn.cursor() as cursor:
        cursor.execute(vector_store_query, query_params)
        results = cursor.fetchall()

        log_ids = set(result["metadata"]["log_id"] for result in results)
        full_logs = []
        for log_id in log_ids:
            full_log = get_full_log(db_conn, log_id)
            full_logs.append(full_log)

    aggregated = aggregation_fn(full_logs, llm=llm, aggregation_query=aggregation_query)
    return aggregated

In [ ]:
job_args = {
    "vector_store_query": "What is AI?",
    "aggregation_query": "Does this conversation contain any sensitive or harmful content?",
    "aggregation_fn": "pairwise_qa",
    "k": 10,
    "distance_threshold": 0.4,
    "filters": {
        "role": "user",
    },
}

results = execute_chat_log_analysis(**job_args, dataset_dir=DATASET_DIR)

In [ ]:
results

# How does this work over RDS?

## 1. Creating 'Custom APIs' like in PySyft
- Alongside RDS, both DO and DS install a `rds-log-analysis` package. this contains utilities and wrappers for submitting standardized jobs
```python
# Example:

import rds_log_analysis

# Client is a subclass of RDSClient, with a few extra methods for easy job submission
client = rds_log_analysis.connect(host="data_owner@openmined.org")
job = client.submit_job(
    vector_store_query="Messages about food and drink",
    ...
)

# get_results loads the result file, and formats it as a Pandas DataFrame
client.get_results(job)
```

## 2. Submitting a job creates 2 files:
- `job.json`: Contains the job arguments ('vector_store_query', ...)
```json
{
    "vector_store_query": "Messages about food and drinks",
    "aggregation_query": "What are the main topics discussed in this conversation?",
    "aggregation_fn": "pairwise_qa",
    "k": 10,
    "distance_threshold": 0.4,
    "filters": {
        "role": "user",
    }
}
```
- `main.py` loads the job args, calls the job function, and writes the result
  - This file is standardized, and not created by the user
```python
from pathlib import Path
import os
from rds_chat_analysis.job import execute_chat_log_analysis
import dotenv

DATA_DIR = os.environ["DATA_DIR"]
OUTPUT_DIR = os.environ["OUTPUT_DIR"]

dotenv.load_dotenv(DATA_DIR / "credentials.env", override=True)

job_args = json.load("./job.json")
result = execute_chat_log_analysis(job_args, DATA_DIR)
result.to_csv(Path(OUTPUT_DIR) / "result.csv", index=False)
```

## 3. Running the job
- Default RDS flow.
- We can do this in a container or on the server.

### 4. DS obtains result
- Default RDS flow.
- We can provide a utility function to load the result file, and format it as a Pandas DataFrame.
